# Demo: Vocabulary Middleware

An agent builds an RDF graph using dataset middleware tools, with vocabulary inspection middleware providing indexed term definitions.

---

- **What is `RDFVocabularyMiddleware`?** A set of tools, prompts, and supporting infrastructure for LangChain agents that use `DatasetMiddleware`. It exposes indexed vocabulary definitions so the Research Agent can discover and inspect standard RDF terms before using them.
- **What problem does it solve for a Research Agent?** The [previous demo](./demo-dataset-middleware.ipynb) showed that agents with dataset tools tend to invent vocabulary rather than reuse established terms. Agents need structured references with which they can dynamically verify their term choices. Vocabulary middleware gives the agent tools to list vocabularies, list terms, and inspect term definitions in native RDF.
- **What will this notebook demonstrate?** Whether giving the agent vocabulary inspection affordances improves term selection compared to the dataset-only condition.

## Where does this notebook fit in the series?

| Condition | Middleware | Notebook |
|-----------|-----------|----------|
| Baseline (prompt-only) | None | [demo-baseline-ontology-extraction.ipynb](./demo-baseline-ontology-extraction.ipynb) |
| Dataset middleware | `DatasetMiddleware` | [demo-dataset-middleware.ipynb](./demo-dataset-middleware.ipynb) |
| **Vocabulary middleware** | `DatasetMiddleware` + `RDFVocabularyMiddleware` | **this notebook** |

## 1. Setup

### 1.1. Bring your own Model

This notebook uses [LangChain's OpenAI integration](https://docs.langchain.com/oss/python/integrations/chat/openai) for [a self-hosted model provider](https://docs.langchain.com/langsmith/playground-model-providers#openai-compatible-endpoint), but you can use any of [LangChain's supported providers](https://docs.langchain.com/oss/python/integrations/providers/all_providers).
What's important is that Deepagents requires a chat model capable of tool invocations.

_NOTE:_ [My Ministral model](https://huggingface.co/mistralai/Ministral-3-14B-Reasoning-2512) recommends adding a specific suffix to its prompt. In this repository we also provide `MinistralPromptSuffixMiddleware` to append it so that it doesn't catch people by surprise if they are using other models. Instead, just modify the prompt directly to match your specific model's requirements. Additionally, the model's documentation indicates that one should use `temperature=1` over `temperature=0`, so we follow that guidance as well.

In [ ]:
import os
import warnings

from langchain_openai import ChatOpenAI

warnings.filterwarnings(
    "ignore",
    message="Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.",
    category=UserWarning,
)

llm = ChatOpenAI(
    model=os.getenv("CHAT_MODEL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    seed=42,
    temperature=1,
)

## 2. The Demonstration

### 2.1. Prompt Setup

The system prompt is assembled from reusable components in [`demo_utils.py`](./demo_utils.py).

Compared to the [dataset middleware demo](./demo-dataset-middleware.ipynb), this notebook:

- Still includes `CORE_PROMPT` (shared modeling guidance) and `STOPPING_CRITERIA` (shared completion rules).
- Still includes `VOCABULARY_TIPS` (tool-agnostic vocabulary-selection guidance).
- Still omits `DATASET_TIPS` because `DatasetMiddleware` provides its own guidance.

What this condition **adds** over the dataset middleware condition:

- `RDFVocabularyMiddleware` tools: `list_vocabularies`, `list_terms`, `describe_term`, `describe_term_spec`.
- `RDFVocabularyMiddleware` prompt guidance encouraging the agent to inspect indexed terms before using them.

What this condition **does not add**:

- No namespace whitelisting or enforcement. The agent is encouraged but not constrained to use indexed terms.

### 2.2. Agent Task Definition

We keep the extraction task itself fixed across the series so that capability changes come from middleware composition rather than from changing the problem statement.

In [ ]:
from typing import Final

from demo_utils import CORE_PROMPT, STOPPING_CRITERIA, VOCABULARY_TIPS

SYSTEM_PROMPT: Final[str] = CORE_PROMPT + VOCABULARY_TIPS + STOPPING_CRITERIA

In [ ]:
from demo_utils import TASK

print(TASK)

#### 2.2.1. Evaluation Contract

The evaluation contract is the same as the previous conditions: parseability first, then set-based overlap metrics against the target graph. Best-practice documentation triples (`rdfs:label`, `rdfs:comment`) are acceptable extras.

In [ ]:
from demo_utils import expected_graph
from rdflib import Graph

print(expected_graph.serialize(format="turtle"))

### 2.3. Tracing the Agent

In [ ]:
from deepagents import create_deep_agent
from langgraph.errors import GraphRecursionError
from rdflib_reasoning.middleware.dataset_middleware import DatasetMiddleware
from rdflib_reasoning.middleware.ministral_middleware import (
    MinistralPromptSuffixMiddleware,
)
from rdflib_reasoning.middleware.rdf_vocabulary_middleware import (
    RDFVocabularyMiddleware,
)
from rdflib_reasoning.middleware.tracing_notebook import LiveNotebookTrace

dataset_middleware = DatasetMiddleware()

with LiveNotebookTrace(heading="Dataset Trace") as trace:
    agent = trace.attach(
        create_deep_agent(
            model=llm,
            system_prompt=SYSTEM_PROMPT,
            middleware=[
                dataset_middleware,
                RDFVocabularyMiddleware(),
                MinistralPromptSuffixMiddleware(),
            ],
        )
    )

    RUN_INPUT = {"messages": [{"role": "user", "content": TASK}]}
    RUN_CONFIG = {"recursion_limit": 40}
    try:
        agent_response = agent.invoke(RUN_INPUT, config=RUN_CONFIG)
    except GraphRecursionError:
        display(trace.sink.snapshot())
        raise

### 2.4. Accessing the Knowledge Base

As in the [dataset middleware demo](./demo-dataset-middleware.ipynb), we retained a reference to the middleware to extract the structured knowledge base without asking the agent to understand RDF serialization formats.

In [ ]:
from rdflib import Dataset

dataset: Dataset = dataset_middleware.session.snapshot_dataset()

print("There are", len(dataset), "triples in the dataset")
print("-" * 80)
print(dataset.serialize(format="turtle"))

actual_graph: Graph = dataset.default_graph

### 2.5. Metrics

The evaluation follows the same protocol as the previous conditions: set-based overlap metrics between the extracted graph and the target graph, with best-practice documentation triples excluded from penalization.

In [ ]:
from demo_utils import evaluate_actual_graph

measures, graphs = evaluate_actual_graph(actual_graph)

intersection = graphs["intersection"]
missing = graphs["missing"]
extra = graphs["extra"]
union = graphs["union"]

#### 2.5.1. Intersection with Target

In [ ]:
import sys

if measures["counts"]["intersection"] > 0:
    print(intersection.serialize(format="turtle"))
else:
    print(
        "The agent's graph output did NOT intersect the expected graph.",
        file=sys.stderr,
    )

In [ ]:
print("Missing target triples:")
print(missing.serialize(format="turtle"))

#### 2.5.2. Extra Facts

Extra triples are not automatically bad. Documentation triples (`rdfs:label`, `rdfs:comment`) are excluded from penalization. Other extras may reveal enriched modeling choices or genuine semantic errors.

In [ ]:
print(extra.serialize(format="turtle"))

#### 2.5.3. Summary Metrics

In [ ]:
from demo_utils import pprint_measures

pprint_measures(measures)

## 3. Conclusions

> What did vocabulary middleware give us?

Vocabulary middleware gave the agent the ability to discover and inspect indexed RDF vocabulary terms before committing to them in `add_triples`.
The middleware prompt encourages the agent to check terms with `describe_term` or `describe_term_spec` before using them, which provides grounding that prompt-only vocabulary guidance cannot.

Compared to the dataset-only condition:

- The agent is expected to make fewer vocabulary misuse errors (e.g., using `rdfs:type` instead of `rdf:type`) because it can verify terms against the index.
- The agent is expected to produce fewer invented properties (e.g., `ex:contains`) because the vocabulary tools provide standard alternatives.
- Residual errors such as plural term names (`ex:Mammals` vs. `ex:Mammal`) persist because those are minting decisions within the open `urn:ex:` namespace, which vocabulary middleware cannot enforce.

> What comes next?

Namespace whitelisting could constrain the agent to allowed vocabulary prefixes and catch non-existent terms in closed namespaces, addressing the residual enforcement gap.
Expanding the indexed vocabulary set (e.g., enabling OWL) would further improve the agent's ability to use richer modeling constructs like `owl:equivalentClass`.